# **Titanic Data Mutual Importance**

This pipeline performs **Mutual Importance Analysis** - a comprehensive feature relationship analysis that goes beyond traditional feature importance. Here's a detailed explanation:

## 🎯 **Core Concept**

Instead of just finding which features predict the target, this analysis examines **how every feature predicts every other feature** in a circular manner.

## 🔄 **Pipeline Workflow**

### 1. **Data Preparation**
- Loads Titanic dataset and drops missing values
- **Label Encoding**: Converts categorical variables to numeric
- **Standard Scaling**: Normalizes all features to same scale

### 2. **Model Training**
- Uses **LightGBM** (gradient boosting) for prediction
- **Cross-validation**: 5-fold CV to ensure robust results
- **Early stopping**: Prevents overfitting

### 3. **Mutual Importance Calculation** ⭐ **Key Innovation**

For **each feature** in the dataset:
- Treat that feature as the **target variable**
- Use **all other features** as predictors
- Train model and record feature importances
- Store results in an **importance matrix**

### 4. **Output: Importance Matrix**
A square matrix where:
- **Rows** = Target features (what we're predicting)
- **Columns** = Predictor features (what we're using to predict)
- **Cell [i,j]** = How important feature j is for predicting feature i

## 📊 **What This Reveals**

### **Traditional Analysis** (one-directional):
```
Features → Target
```

### **Mutual Analysis** (bi-directional):
```
Feature A → Feature B
Feature B → Feature A  
Feature A → Target
Target → Feature A
```

## 🔍 **Key Insights You Get**

### 1. **Forward Relationships**
- How features predict the target (normal feature importance)

### 2. **Reverse Relationships** 
- How the target predicts features (reveals leakage potential)

### 3. **Feature Dependencies**
- Which features are highly interdependent
- Redundant or collinear features

### 4. **Central Features**
- Features that appear important across many predictions

## 🛠 **Visualization Tools**

### 1. **Heatmap**
- Visual representation of the entire importance matrix
- Shows patterns and clusters of related features

### 2. **Top Relationships**
- Bar chart of strongest feature→feature relationships
- Identifies the most influential connections

### 3. **Connectivity Analysis**
- Measures how "central" each feature is in the network
- High connectivity = influential feature in the system

## 💡 **Practical Applications**

### **Feature Selection**
- Remove redundant features that provide similar information
- Keep features that uniquely contribute to predictions

### **Data Leakage Detection**
- If target strongly predicts a feature, potential leakage exists

### **Domain Understanding**
- Discover hidden relationships in your data
- Understand the causal-like structure of variables

### **Model Interpretation**
- Go beyond "feature X is important" to understand WHY it's important

## ⚠️ **Important Notes**

- **Computationally intensive**: Requires training n models for n features
- **Correlation ≠ Causation**: Shows predictive relationships, not necessarily causal
- **Scale dependent**: Results can vary with different preprocessing

This approach provides a **holistic view** of feature relationships that traditional methods miss, making it invaluable for understanding complex datasets and building robust models.

In [ ]:
# Mutual Importance Analysis for All Features
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from tqdm import tqdm
import lightgbm as lgbm
import joblib
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

# ============================================================================
# Data Preparation
# ============================================================================
data0 = pd.read_csv("/kaggle/input/titanic/train.csv")
data0 = data0.dropna()
print(len(data0))


# Label encode categorical columns
def labelencoder(df):
    for c in df.columns:
        if df[c].dtype == 'object': 
            df[c] = df[c].fillna('N')
            lbl = LabelEncoder()
            lbl.fit(list(df[c].values))
            df[c] = lbl.transform(df[c].values)
    return df

from sklearn.preprocessing import StandardScaler

data1a = labelencoder(data0)
display(data1a)

scaler = StandardScaler()
data1 = pd.DataFrame(
    scaler.fit_transform(data1a),
    columns=data1a.columns,
    index=data1a.index
)
display(data1)

# ============================================================================
# Model Training Function
# ============================================================================
def fit_lgbm(X, y, cv, params: dict = None, verbose: int = 50):
    """Train LightGBM model with cross-validation"""
    if params is None:
        params = {}

    models = []
    oof_pred = np.zeros_like(y, dtype=float)

    for i, (idx_train, idx_valid) in enumerate(cv): 
        x_train, y_train = X[idx_train], y[idx_train]
        x_valid, y_valid = X[idx_valid], y[idx_valid]

        clf = lgbm.LGBMRegressor(**params)

        clf.fit(x_train, y_train, 
                eval_set=[(x_valid, y_valid)],
                callbacks=[lgbm.early_stopping(stopping_rounds=50, verbose=False)])

        pred_i = clf.predict(x_valid)
        oof_pred[idx_valid] = pred_i
        models.append(clf)
        print(f'Fold {i} RMSE: {mean_squared_error(y_valid, pred_i) ** .5:.4f}')

    score = mean_squared_error(y, oof_pred) ** .5
    print(f'Overall RMSE: {score:.4f}\n')
    return oof_pred, models

# ============================================================================
# Mutual Importance Calculation
# ============================================================================
def calculate_mutual_importance(data, target_col=None, n_splits=5, random_state=42):
    """
    Calculate mutual importance matrix for all features
    
    Args:
        data: DataFrame with all features
        target_col: Optional target column name to exclude from predictors
        n_splits: Number of CV folds
        random_state: Random seed
        
    Returns:
        importance_matrix: DataFrame where [i,j] represents importance of feature j 
                          when predicting feature i
    """
    
    # LightGBM parameters
    params = {
        'objective': 'rmse', 
        'learning_rate': 0.1,
        'reg_lambda': 1.0,
        'reg_alpha': 0.1,
        'max_depth': 5, 
        'n_estimators': 500, 
        'colsample_bytree': 0.5, 
        'min_child_samples': 10,
        'subsample_freq': 3,
        'subsample': 0.9,
        'importance_type': 'gain', 
        'random_state': random_state,
        'num_leaves': 31,
        'verbose': -1
    }
    
    # Prepare columns
    if target_col and target_col in data.columns:
        columns = [col for col in data.columns if col != target_col]
    else:
        columns = data.columns.tolist()
    
    # Initialize importance matrix
    importance_matrix = pd.DataFrame(
        np.zeros((len(columns), len(columns))),
        index=columns,
        columns=columns
    )
    
    # Calculate importance for each feature as target
    for target_feature in tqdm(columns, desc="Calculating mutual importance"):
        # Prepare data: use all other features to predict target_feature
        predictor_cols = [col for col in columns if col != target_feature]
        X = data[predictor_cols].values
        y = data[target_feature].values
        
        # Cross-validation
        fold = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        cv = list(fold.split(X, y))
        
        # Train model
        print(f"\n{'='*60}")
        print(f"Target Feature: {target_feature}")
        print(f"{'='*60}")
        oof, models = fit_lgbm(X, y, cv, params=params)
        
        # Average feature importance across folds
        avg_importance = np.mean([model.feature_importances_ for model in models], axis=0)
        
        # Store in matrix
        importance_matrix.loc[target_feature, predictor_cols] = avg_importance
    
    return importance_matrix

# ============================================================================
# Visualization Functions
# ============================================================================
def visualize_importance_heatmap(importance_matrix, figsize=(14, 12), cmap='viridis'):
    """Visualize mutual importance matrix as heatmap"""
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Create heatmap
    sns.heatmap(importance_matrix, 
                annot=False,
                fmt='.2f',
                cmap=cmap,
                cbar_kws={'label': 'Feature Importance'},
                ax=ax)
    
    ax.set_title('Mutual Feature Importance Matrix\n(Row: Target Feature, Column: Predictor Feature)', 
                 fontsize=14, pad=20)
    ax.set_xlabel('Predictor Features', fontsize=12)
    ax.set_ylabel('Target Features', fontsize=12)
    
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    return fig, ax

def visualize_top_relationships(importance_matrix, top_n=20):
    """Visualize top N feature relationships"""
    
    # Get all pairwise relationships (excluding diagonal)
    relationships = []
    for target in importance_matrix.index:
        for predictor in importance_matrix.columns:
            if target != predictor:
                relationships.append({
                    'target': target,
                    'predictor': predictor,
                    'importance': importance_matrix.loc[target, predictor]
                })
    
    # Sort and get top N
    df_relationships = pd.DataFrame(relationships)
    df_relationships = df_relationships.sort_values('importance', ascending=False).head(top_n)
    
    # Create labels
    df_relationships['pair'] = df_relationships['predictor'] + ' → ' + df_relationships['target']
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.3)))
    
    sns.barplot(data=df_relationships, 
                y='pair', 
                x='importance',
                palette='rocket',
                ax=ax)
    
    ax.set_title(f'Top {top_n} Feature Relationships by Importance', fontsize=14, pad=15)
    ax.set_xlabel('Feature Importance', fontsize=12)
    ax.set_ylabel('Feature Relationship', fontsize=12)
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    
    return fig, ax, df_relationships

def analyze_feature_connectivity(importance_matrix, threshold=None):
    """Analyze how connected each feature is to others"""
    
    if threshold is None:
        threshold = importance_matrix.values[importance_matrix.values > 0].mean()
    
    # Count strong connections for each feature
    # As predictor (how many features depend on it)
    as_predictor = (importance_matrix > threshold).sum(axis=0)
    # As target (how many features it depends on)
    as_target = (importance_matrix > threshold).sum(axis=1)
    
    connectivity_df = pd.DataFrame({
        'Feature': importance_matrix.index,
        'As_Predictor': as_predictor.values,
        'As_Target': as_target.values,
        'Total_Connections': as_predictor.values + as_target.values
    }).sort_values('Total_Connections', ascending=False)
    
    return connectivity_df

# ============================================================================
# Main Analysis
# ============================================================================
if __name__ == "__main__":
    
    # Calculate mutual importance INCLUDING 'target'
    print("\n" + "="*70)
    print("STARTING MUTUAL IMPORTANCE ANALYSIS (INCLUDING 'target')")
    print("="*70 + "\n")
    
    print("⚠ NOTE: 'target' is included in the analysis.")
    print("This will reveal:")
    print("  1. How features predict 'target' (forward direction)")
    print("  2. How 'target' predicts features (reverse direction)")
    print("  3. Hidden relationships between all variables\n")
    
    importance_matrix = calculate_mutual_importance(
        data1, 
        target_col=None,  # Include ALL columns including 'target'
        n_splits=3,  # Use fewer splits for faster computation
        random_state=42
    )
    
    # Save results
    os.makedirs('mutual_importance_results', exist_ok=True)
    importance_matrix.to_csv('mutual_importance_results/importance_matrix.csv')
    print("\n✓ Importance matrix saved to 'mutual_importance_results/importance_matrix.csv'")
    
    # Visualize heatmap
    print("\n" + "="*70)
    print("CREATING VISUALIZATIONS")
    print("="*70 + "\n")
    
    fig1, ax1 = visualize_importance_heatmap(importance_matrix)
    fig1.savefig('mutual_importance_results/importance_heatmap.png', dpi=150, bbox_inches='tight')
    print("✓ Heatmap saved to 'mutual_importance_results/importance_heatmap.png'")
    
    # Visualize top relationships
    fig2, ax2, top_rels = visualize_top_relationships(importance_matrix, top_n=100)
    fig2.savefig('mutual_importance_results/top_relationships.png', dpi=150, bbox_inches='tight')
    print("✓ Top relationships saved to 'mutual_importance_results/top_relationships.png'")
    
    # Analyze connectivity
    connectivity = analyze_feature_connectivity(importance_matrix)
    connectivity.to_csv('mutual_importance_results/feature_connectivity.csv', index=False)
    print("✓ Connectivity analysis saved to 'mutual_importance_results/feature_connectivity.csv'")
    
    # Display summary statistics
    print("\n" + "="*70)
    print("SUMMARY STATISTICS")
    print("="*70)
    print(f"\nMean importance: {importance_matrix.values.mean():.4f}")
    print(f"Max importance: {importance_matrix.values.max():.4f}")
    print(f"Std importance: {importance_matrix.values.std():.4f}")
    
    print("\n" + "="*70)
    print("TOP 10 MOST CONNECTED FEATURES")
    print("="*70)
    print(connectivity.head(10).to_string(index=False))
    
    print("\n" + "="*70)
    print("TOP 10 FEATURE RELATIONSHIPS")
    print("="*70)
    print(top_rels.head(10).to_string(index=False))
    
    plt.show()

### The presented pairs with high importance will lead to ideas for new features.